# 📝 통계 기초 과제 LV3 정답 — 고객 데이터 기술통계·추론 리포트 (강사용)

각 단계의 **모범 코드 + 자가채점 + 해설** 입니다.

In [ ]:
# [제공 코드] 통계 분석에 쓸 라이브러리와 한글 폰트를 준비합니다.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지
sns.set_theme(font=KOREAN_FONT, rc={"axes.unicode_minus": False})

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을, `describe()` 로 수치·범주 요약을 봅니다. (아래 셀은 실행만 하면 됩니다.)

이 데이터는 한 유통사의 고객 2,240명 기록입니다. 소득(`Income`)에 결측이 있고, 결혼상태(`Marital_Status`)에 정상 범주가 아닌 오염값(`Absurd`·`YOLO`·`Alone`)이 섞여 있어 **정제가 필요한 실전 데이터**입니다.

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·수치/범주 요약
#   (미리보기 전용 변수 preview 를 씁니다. 문제 풀이용 df 는 1단계에서 직접 불러오세요.)
preview = pd.read_csv("../../day08_기술통계_추론통계/data/marketing_campaign.csv")
print("행·열 크기:", preview.shape)
print("\n[앞 5행] head()"); display(preview.head())
print("\n[열·자료형·결측] info()"); preview.info()
print("\n[수치 요약] describe()"); display(preview.describe())
print("\n[범주 요약] describe(exclude='number')"); display(preview.describe(exclude="number"))

## 1. 고객 기술통계 리포트
**배경**: 마케팅팀이 고객의 **소득과 지출**을 한 장으로 요약해 달라고 요청했습니다. 원본에는 결측과 오염값이 있으므로 먼저 **정제**한 뒤, 대표값·산포·분포·상관·신뢰구간까지 기술통계로 리포트를 완성합니다.

아래 각 `### N단계` 셀의 지시대로 **하나의 `df` 를 이어서** 분석합니다(2단계 정제·3단계 파생 컬럼이 뒤로 이어집니다).

**최종 목표(자가채점 기준)**
| 단계 | 확인 항목 |
| --- | --- |
| 1단계 | 원본 `(2240, 29)`, `Income` 결측 24개, `Marital_Status` 8범주(오염 포함) |
| 2단계 | 정제 후 행수 2233, `Income` 결측 0, 오염 결혼상태 0 |
| 3단계 | `total_spend` 평균 605.9, `age` 중앙값 44 |
| 4단계 | `Income` 평균 52234.71·중앙 51381.5·표준편차 25062.79·CV 47.98·이상치 8개 |
| 5단계 | 총지출 분포 히스토그램(왜도 표기) — 완성 그래프처럼 |
| 6단계 | 수치 4열 상관행렬 히트맵 — 완성 그래프처럼 |
| 7단계 | `Income` 평균 95% 신뢰구간 [51195.19, 53274.23] |
| 8단계 | 인사이트 서술(3문장 이상) |

### 1단계 — 데이터 불러오기·구조 파악
`../../day08_기술통계_추론통계/data/marketing_campaign.csv` 를 `df` 로 불러오고, `df.shape`, `df["Income"].isna().sum()`(소득 결측 수), `df["Marital_Status"].value_counts()`(결혼상태 범주별 개수) 를 출력하세요.

- **요구사항**: 원본은 `(2240, 29)` 이고, `Income` 결측은 **24개**, `Marital_Status` 는 정상 6범주에 오염값 `Absurd`·`YOLO`·`Alone` 이 섞여 **총 8범주**입니다.
- **주의**: 아직 정제하지 않은 **원본 그대로**의 값을 확인하는 단계입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 파일을 데이터프레임으로 읽고 크기·결측 수·범주별 개수를 각각 출력한다.

세부구현:
1. read_csv 로 데이터를 df 에 담는다
2. shape 로 행·열 크기를 출력한다
3. Income 열의 결측 개수(isna 합)를 출력한다
4. Marital_Status 의 범주별 개수를 출력한다
```

</details>

In [ ]:
df = pd.read_csv("../../day08_기술통계_추론통계/data/marketing_campaign.csv")
print(df.shape)
# 분석 전에 결측과 이상한 범주부터 확인한다 — 이 둘을 놓치면 뒤의 모든 통계가 조용히 틀어진다.
print(df["Income"].isna().sum())
print(df["Marital_Status"].value_counts())

In [ ]:
# [자가채점]
assert df.shape == (2240, 29)
assert int(df["Income"].isna().sum()) == 24
assert df["Marital_Status"].nunique() == 8
print("✅ 1단계 통과!")

### 해설 — 문제 1 · 1단계
- **접근법**: `read_csv` 로 읽은 뒤 `shape`(크기)·`isna().sum()`(결측)·`value_counts()`(범주 분포) 세 가지로 데이터의 상태를 먼저 진단합니다. 분석 전에 **무엇이 문제인지 아는** 단계예요.
- **흔한 실수**: `value_counts()` 와 `nunique()` 는 기본적으로 **결측을 세지 않습니다**. 결측까지 범주로 보려면 `value_counts(dropna=False)` 를 쓰세요.
- **대안**: 열이 많을 때는 `info()` 한 줄이면 열별 결측 수와 자료형을 한눈에 볼 수 있어 더 빠릅니다.

### 2단계 — 정제 (결측 대체 · 오염값 제거)
같은 `df` 를 다음 **순서**로 정제하세요.

1. **소득 결측 대체**: `Income` 의 결측을 `Income` 열의 **중앙값**(`df["Income"].median()`)으로 채웁니다.
2. **오염값 제거**: `Marital_Status` 가 `Absurd`·`YOLO`·`Alone` 인 행을 **삭제**합니다.

- **요구사항**: 정제 후 `df` 의 **행수는 2233**, `Income` 결측은 **0**, 오염 결혼상태는 **0개**여야 합니다.
- **주의**: 결측을 먼저 채운 **뒤** 행을 삭제하세요(순서가 중앙값에 영향). 삭제 후 `df` 를 그대로 이어 씁니다.
- **처음이라도 괜찮아요**: 여기 쓰는 `fillna`·`median`·`isin` 은 앞선 pandas 단원에서 익힌 기능입니다. 낯설면 아래 힌트를 펼쳐 네 단계를 그대로 따라오세요.

<details><summary>힌트</summary>

```text
접근방법:
- 정제는 '결측 채우기 → 오염 행 버리기' 두 단계다. 순서를 지켜 결측을 먼저 메운 뒤 행을 삭제한다.
- fillna 는 결측(NaN)을 괄호 안에 넣은 값으로 바꿔 준다. median 으로 그 열의 중앙값을 먼저 구해 fillna 에 넘긴다.
- isin 은 값이 목록 안에 있으면 참을 돌려준다. 그 앞에 물결표(~)를 붙이면 '목록에 없는' 행만 남길 수 있다.

세부구현:
1. Income 열의 중앙값을 median 으로 구한다
2. 그 중앙값을 fillna 에 넘겨 Income 의 결측을 채우고 결과를 다시 Income 열에 넣는다
3. 삭제할 오염값(Absurd·YOLO·Alone)을 리스트로 모은다
4. Marital_Status 가 그 목록에 드는지 isin 으로 표시하고, 앞에 물결표(~)를 붙여 목록에 없는 행만 골라 df 에 다시 담는다
```

</details>

In [ ]:
# 평균이 아니라 중앙값으로 채운다 — 소득처럼 오른쪽 꼬리가 긴 분포에서는 평균이 소수의 고소득에 끌려간다.
df["Income"] = df["Income"].fillna(df["Income"].median())
bad_status = ["Absurd", "YOLO", "Alone"]
df = df[~df["Marital_Status"].isin(bad_status)]
print(df.shape)

In [ ]:
# [자가채점]
assert df.shape[0] == 2233
assert int(df["Income"].isna().sum()) == 0
assert df["Marital_Status"].isin(["Absurd", "YOLO", "Alone"]).sum() == 0
print("✅ 2단계 통과!")

### 해설 — 문제 1 · 2단계
- **접근법**: `fillna(중앙값)` 으로 결측을 메운 **뒤** `isin` + `~` 로 오염 행을 걸러냅니다. 순서가 중요해요 — 행을 먼저 지우면 중앙값 자체가 달라져 결과가 바뀝니다.
- **흔한 실수**: `df[~조건]` 의 결과를 `df` 에 **다시 대입하지 않아** 원본이 그대로인 경우가 가장 많습니다. 필터링은 새 데이터프레임을 돌려줄 뿐 원본을 바꾸지 않습니다.
- **대안**: 결측 대체는 중앙값 말고 평균·최빈값·그룹별 중앙값도 있습니다. 소득처럼 오른쪽으로 치우친 변수는 평균보다 **중앙값이 안전**합니다(이상치에 덜 끌려감).

### 3단계 — 파생 변수 만들기 (총지출 · 나이)
정제된 `df` 에 두 파생 컬럼을 추가하세요.

- `total_spend`: 6개 지출 열(['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']) 의 **행별 합** — 고객 한 명의 총지출.
- `age`: **기준연도 2014** 에서 출생연도를 뺀 값 → `2014 - df["Year_Birth"]`.

- **요구사항**: `total_spend` 의 평균은 약 **605.9**, `age` 의 중앙값은 **44** 입니다.
- **주의**: 6개 지출 열의 합은 `df[열목록].sum(axis=1)` 처럼 **행 방향(axis=1)** 으로 더합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 새 컬럼은 df 의 새 이름 자리에 계산 결과를 대입해 추가한다.
- 여러 열을 '한 사람(행)별로' 더하려면 sum 에 axis 를 1(행 방향)로 준다. axis 를 빼면 열마다 세로로 더해 버리니 주의.

세부구현:
1. 지출 6열의 이름을 리스트로 모은다
2. 그 열들을 골라 sum 에 axis 1 을 주어 행마다 더해 total_spend 컬럼에 넣는다
3. 2014 에서 Year_Birth 를 빼 age 컬럼에 넣는다
```

</details>

In [ ]:
mnt_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
# axis=1 은 '행 방향으로 더하기' — 고객 한 명의 품목별 지출을 합쳐 한 값으로 만든다.
df["total_spend"] = df[mnt_cols].sum(axis=1)
df["age"] = 2014 - df["Year_Birth"]
print(round(df["total_spend"].mean(), 2), df["age"].median())

In [ ]:
# [자가채점]  통계값은 정확일치 대신 허용오차(abs<tol)로 비교합니다.
assert "total_spend" in df.columns and "age" in df.columns
assert abs(float(df["total_spend"].mean()) - 605.9) < 0.01
assert int(df["age"].median()) == 44
print("✅ 3단계 통과!")

### 해설 — 문제 1 · 3단계
- **접근법**: 여러 열의 **행별 합**은 `df[열목록].sum(axis=1)` 입니다. `axis=1` 이 '가로로, 한 사람 안에서 더하기'라는 뜻이에요.
- **흔한 실수**: `axis` 를 빼면 기본값 0 이라 **열별 합계**(전체 고객의 합)가 나옵니다. 결과 길이가 6이면 방향을 잘못 잡은 것입니다.
- **대안**: 나이는 **기준연도를 고정**해야 재현됩니다. 오늘 날짜로 계산하면 해가 바뀔 때 자가채점이 깨집니다.

### 4단계 — 대표값·산포 (소득) 
`Income` 의 대표값과 산포를 구해 아래 이름의 변수에 담으세요.

- `inc_mean` = 평균(`mean`), `inc_median` = 중앙값(`median`), `inc_std` = 표준편차(**표본, `std(ddof=1)`**)
- `inc_cv` = **변동계수**(%) = `inc_std / inc_mean * 100`
- `inc_outliers` = **1.5×IQR 규칙 이상치 개수**(정수) — Q1·Q3 은 `quantile(0.25)`·`quantile(0.75)`, IQR = Q3−Q1, 경계 밖(`< Q1-1.5*IQR` 또는 `> Q3+1.5*IQR`) 개수.

- **요구사항(반올림 자리)**: `inc_mean`→2자리 52234.71, `inc_median`→2자리 51381.5, `inc_std`→2자리 25062.79, `inc_cv`→2자리 47.98, `inc_outliers`→정수 8.
- **주의**: 평균이 중앙값보다 큰 것은 **오른쪽 꼬리(고소득 이상치)** 때문입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 평균·중앙·표준편차를 구하고, 변동계수는 표준편차를 평균으로 나눠 100 을 곱한다.
- 이상치는 사분위와 IQR 로 위아래 경계를 만들어 그 밖의 개수를 센다.

세부구현:
1. Income 의 평균·중앙값·표준편차(ddof=1)를 각각 변수에 담는다
2. 변동계수 = 표준편차 / 평균 * 100
3. 1사분위·3사분위를 구해 IQR 과 아래·위 경계를 만든다
4. 경계 밖에 있는 값의 개수를 세어 정수로 담는다
```

</details>

In [ ]:
inc = df["Income"]
inc_mean = inc.mean()
inc_median = inc.median()
inc_std = inc.std(ddof=1)
# 평균·중앙값·CV·이상치를 한 번에 낸다 — 하나만 보면 분포의 모양을 잘못 읽는다.
inc_cv = inc_std / inc_mean * 100
q1 = inc.quantile(0.25)
q3 = inc.quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr
inc_outliers = int(((inc < low) | (inc > high)).sum())
print(round(inc_mean, 2), round(inc_median, 2), round(inc_std, 2), round(inc_cv, 2), inc_outliers)

In [ ]:
# [자가채점]
assert abs(float(inc_mean) - 52234.71) < 0.01
assert abs(float(inc_median) - 51381.5) < 0.01
assert abs(float(inc_std) - 25062.79) < 0.01
assert abs(float(inc_cv) - 47.98) < 0.01
assert inc_outliers == 8
print("✅ 4단계 통과!")

### 해설 — 문제 1 · 4단계
- **접근법**: 평균·중앙값으로 중심을, 표준편차로 퍼짐을, **변동계수(CV)** 로 단위에 의존하지 않는 상대적 퍼짐을, 1.5×IQR 로 이상치 개수를 봅니다.
- **흔한 실수**: `std()` 의 기본 자유도는 pandas 가 `ddof=1`(표본), numpy 는 `ddof=0`(모집단)으로 **다릅니다**. 문제는 표본 기준이므로 `ddof=1` 을 명시하세요.
- **대안**: 평균(52234.71) > 중앙값(51381.5) 이면 오른쪽 꼬리가 있다는 신호입니다. 이상치가 신경 쓰이면 절사평균(`stats.trim_mean`)도 함께 보세요.

### 5단계 — 총지출 분포 (히스토그램 + 왜도)
`total_spend` 의 분포를 **히스토그램**으로 그리고, 분포의 비대칭을 나타내는 **왜도(skew)** 값을 제목에 표기하세요.

- `skew_val` = `stats.skew(df["total_spend"])` (약 0.86 — 오른쪽으로 꼬리가 긴 분포).
- `sns.histplot(data=df, x="total_spend", bins=40)` 로 그리고, 제목에 왜도 값을 넣습니다(예: `총지출 분포 — 왜도 0.86`).
- 그래프가 겹치지 않도록 그리기 직전에 `plt.figure()` 를 호출하세요.

이 단계는 **자가채점이 없습니다** — 아래 **완성 그래프(정답)** 와 같은 모양으로 그리면 됩니다.

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day08_기술통계_추론통계/images/과제/lv3_q1_s5.png" width="560"/>

In [ ]:
plt.figure(figsize=(8, 5))
# 왜도를 제목에 함께 넣어 그림과 숫자를 같이 본다 — 오른쪽 꼬리가 길면 양수로 나온다.
skew_val = stats.skew(df["total_spend"])
ax = sns.histplot(data=df, x="total_spend", bins=40)
ax.set_title(f"총지출 분포 — 왜도 {skew_val:.2f}")
ax.set_xlabel("총지출(total_spend)")
ax.set_ylabel("고객 수")
plt.show()

### 해설 — 문제 1 · 5단계
- **접근법**: `histplot` 으로 분포의 **모양**을 보고 `stats.skew` 로 그 치우침을 **수치**로 못 박습니다. 그림과 숫자를 나란히 제시하는 것이 리포트의 기본형이에요.
- **흔한 실수**: `plt.figure()` 없이 여러 번 그리면 앞 그림 위에 겹쳐 그려집니다. `bins` 를 너무 작게 잡으면 분포 모양이 뭉개집니다.
- **대안**: 왜도가 큰 분포는 로그 변환 후 다시 그려 보면 가려져 있던 구조가 드러나는 일이 많습니다.

### 6단계 — 상관행렬 히트맵 (수치 4열)
소득·지출·구매횟수가 서로 어떻게 움직이는지 **상관행렬 히트맵**으로 보세요.

- 대상 4열: `["Income", "total_spend", "NumWebPurchases", "NumStorePurchases"]`
- `corr = df[대상4열].corr()` 로 상관행렬을 만들고, `sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)` 로 그립니다.
- 그리기 직전에 `plt.figure()` 를 호출하세요.

이 단계도 **자가채점이 없습니다** — 아래 **완성 그래프(정답)** 처럼 그리면 됩니다.

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day08_기술통계_추론통계/images/과제/lv3_q1_s6.png" width="520"/>

In [ ]:
plt.figure(figsize=(6, 5))
corr_cols = ["Income", "total_spend", "NumWebPurchases", "NumStorePurchases"]
corr = df[corr_cols].corr()
# vmin·vmax 를 -1~1 로 고정한다 — 데이터에 맞춰 색 범위가 바뀌면 다른 그림과 비교할 수 없다.
ax = sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
ax.set_title("주요 수치 변수 상관행렬")
plt.show()

### 해설 — 문제 1 · 6단계
- **접근법**: 관심 있는 **수치 열만 골라** `corr()` 로 상관행렬을 만들고 `heatmap(annot=True)` 로 값을 찍습니다.
- **흔한 실수**: 전체 `df` 로 `corr()` 하면 ID·이진 플래그까지 섞여 표가 읽히지 않습니다. 볼 열을 먼저 추리는 것이 핵심입니다.
- **대안**: `cmap='coolwarm'` 에 `vmin=-1, vmax=1` 을 주면 양·음 상관의 강도를 색으로 바로 비교할 수 있습니다.

### 7단계 — 소득 평균의 95% 신뢰구간
`Income` **평균**의 **95% 신뢰구간**을 정규근사 공식으로 구해 `ci_low`·`ci_high` 에 담으세요.

- 신뢰구간은 **표본평균을 중심으로 표준오차만큼 좌우로 벌린 범위**입니다. 표본이 크므로 **정규근사**를 씁니다 — `scipy.stats` 에서 표준오차를 구하는 함수와, 신뢰수준·평균·표준오차를 받아 (하한, 상한)을 돌려주는 정규분포 구간 함수를 찾아 쓰세요.
- **주의**: `scale` 에 표준편차가 아니라 **표준오차**를 넣어야 평균의 신뢰구간이 됩니다.
- **요구사항(2자리 반올림)**: `ci_low` = **51195.19**, `ci_high` = **53274.23**.
- **주의**: 여기서는 신뢰구간 공식만 사용합니다(표본이 크므로 정규근사).

<details><summary>힌트</summary>

```text
접근방법:
- 평균과 표준오차를 구한 뒤, 정규분포 구간 함수에 신뢰수준·평균·표준오차를 넣는다.

세부구현:
1. Income 의 평균을 구한다
2. 표준오차를 sem 으로 구한다
3. norm.interval 에 0.95 와 평균·표준오차를 넣어 하한·상한을 받는다
4. 하한을 ci_low, 상한을 ci_high 에 담는다
```

</details>

In [ ]:
inc = df["Income"]
ci_low, ci_high = stats.norm.interval(0.95, loc=inc.mean(), scale=stats.sem(inc))
print(round(ci_low, 2), round(ci_high, 2))

In [ ]:
# [자가채점]
assert abs(float(ci_low) - 51195.19) < 0.01
assert abs(float(ci_high) - 53274.23) < 0.01
print("✅ 7단계 통과!")

### 해설 — 문제 1 · 7단계
- **접근법**: `stats.norm.interval(0.95, loc=표본평균, scale=stats.sem(x))` 로 모평균의 95% 신뢰구간을 구합니다.
- **흔한 실수**: `scale` 에 **표준편차(std)를 넣는 실수**가 가장 잦습니다. 들어갈 값은 **표준오차(SE = std/√n)** 이고 `stats.sem` 이 그것을 돌려줍니다.
- **대안**: 표본이 작을 때는 t분포(`stats.t.interval(0.95, df=n-1, ...)`)를 씁니다. 여기 n 은 2233 이라 두 구간이 거의 같지만 **소수 둘째 자리에서 갈리므로**(norm 51195.19 vs t 51194.62) 자가채점은 `norm` 기준입니다.

### 8단계 — 인사이트 (서술)
위 대표값·산포·분포·상관·신뢰구간을 근거로 **고객의 소득과 지출 특징**을 **3문장 이상** 서술하세요.
- 평균과 중앙값의 관계(왜도), 이상치의 의미, 소득과 지출·구매의 상관을 말로 풀어 보세요.

**인사이트 (모범 서술)**

고객 소득은 평균 52,235 로 중앙값 51,382 보다 높고 총지출의 왜도가 0.86 으로 **오른쪽 꼬리가 긴 분포**입니다 — 소수의 고소득·고지출 고객(1.5×IQR 이상치 8명)이 평균을 끌어올리고 있어, 대표값으로는 중앙값이 더 안전합니다. 상관행렬을 보면 소득과 총지출의 상관이 0.66 으로 뚜렷하고, 총지출은 매장 구매횟수(0.67)·웹 구매횟수(0.52)와도 함께 움직여 **잘 버는 고객이 더 많이 쓰고 더 자주 산다**는 패턴이 읽힙니다. 소득 평균의 95% 신뢰구간이 [51195, 53274] 로 좁게 잡혀, 표본이 커 평균 추정이 안정적임을 알 수 있습니다.

## 2. 지출·반응 심화 리포트
**배경**: 이번에는 **어떤 고객이 더 많이 쓰고, 캠페인에 반응하는지** 를 집단별로 비교합니다. 교육수준·결혼상태별 지출 차이를 보고, 마지막 캠페인 반응(`Response`) 여부에 따른 지출 차이와, 총지출과 캠페인 수락의 관계를 기술통계로 정리합니다.

**최종 목표(자가채점 기준)**
| 단계 | 확인 항목 |
| --- | --- |
| 1단계 | 정제·파생 재현: 행수 2233, `total_spend` 평균 605.9 |
| 2단계 | 교육수준별 평균 — PhD 총지출 674.73·소득 56169.94, Basic 총지출 81.8 |
| 3단계 | 결혼상태별 총지출 박스플롯 — 완성 그래프처럼 |
| 4단계 | 반응별 총지출 평균 — 미반응 538.85·반응 991.24, 차이 452.39 |
| 5단계 | 총지출 ↔ 캠페인 수락 합 상관 r 0.46 |
| 6단계 | 인사이트 서술 |

### 1단계 — 불러오기·정제·파생 (문제 1 방식 재사용)
문제 2 를 위해 데이터를 **처음부터 다시** 불러와 문제 1 과 **같은 방식**으로 정제·파생합니다.

1. `../../day08_기술통계_추론통계/data/marketing_campaign.csv` 를 `df` 로 불러온다.
2. `Income` 결측을 중앙값으로 채운다.
3. `Marital_Status` 가 `Absurd`·`YOLO`·`Alone` 인 행을 제거한다.
4. `total_spend` = 6개 지출 열(['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']) 의 행별 합 을 만든다.

- **요구사항**: 정제 후 행수 **2233**, `total_spend` 평균 약 **605.9**.
- **주의**: 문제 1 에서 만든 `df` 에 이어 쓰지 말고 **새로 로드**하세요(문제 간 오염 방지).

<details><summary>힌트</summary>

```text
접근방법:
- 문제 1 의 2·3단계에서 한 정제·파생을 그대로 다시 한다: fillna 로 결측 채우기 → isin 을 물결표(~)로 부정해 오염 행 버리기 → 지출 6열 합.

세부구현:
1. read_csv 로 원본을 새로 읽어 df 에 담는다
2. Income 의 중앙값(median)을 fillna 에 넘겨 결측을 채운다
3. Marital_Status 가 오염 목록에 없는 행만 isin 과 물결표(~)로 남긴다
4. 지출 6열을 골라 sum 에 axis 1 을 주어 행마다 더해 total_spend 를 만든다
```

</details>

In [ ]:
# 2부는 따로 실행해도 되도록 1부의 정제 과정을 한 셀에 다시 모아 둔 것이다.
#  (실무 노트북도 이렇게 "여기서부터 시작해도 되는 지점"을 만들어 두면 편하다.)
df = pd.read_csv("../../day08_기술통계_추론통계/data/marketing_campaign.csv")
df["Income"] = df["Income"].fillna(df["Income"].median())
df = df[~df["Marital_Status"].isin(["Absurd", "YOLO", "Alone"])]
mnt_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
df["total_spend"] = df[mnt_cols].sum(axis=1)
print(df.shape, round(df['total_spend'].mean(), 2))

In [ ]:
# [자가채점]
assert "age" not in df.columns, "문제 1 의 df 를 이어 쓰지 말고 원본을 새로 불러오세요"
assert df.shape[0] == 2233
assert abs(float(df["total_spend"].mean()) - 605.9) < 0.01
print("✅ 1단계 통과!")

### 해설 — 문제 2 · 1단계
- **접근법**: 문제 1 의 정제·파생을 **그대로 반복**합니다. 같은 순서를 지켜야 같은 숫자가 재현됩니다.
- **흔한 실수**: 문제 1 에서 쓰던 `df` 를 그대로 이어 쓰면 **이미 정제된 상태**라 이중 처리가 됩니다. 반드시 원본을 다시 읽어 시작하세요.
- **대안**: 이렇게 반복되는 전처리는 함수로 묶어 두면(4일차 함수 단원) 재사용이 쉽고 실수도 줄어듭니다.

### 2단계 — 교육수준별 소득·지출 비교 (groupby)
교육수준(`Education`)별로 **평균 소득과 평균 총지출**을 집계해 `edu_stats` 에 담으세요.

- `Education` 으로 묶어 `Income`·`total_spend` 두 열의 **평균**을 구해 `edu_stats` 에 담으세요 — 행은 교육수준(5범주), 열은 그 두 개라 모양은 `(5, 2)` 입니다.

- **요구사항(2자리 반올림)**: `edu_stats.loc["PhD", "total_spend"]` = **674.73**, `edu_stats.loc["PhD", "Income"]` = **56169.94**, `edu_stats.loc["Basic", "total_spend"]` = **81.8**.
- **주의**: 교육수준은 5범주(`Basic`·`2n Cycle`·`Graduation`·`Master`·`PhD`) 이므로 `edu_stats` 모양은 `(5, 2)` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 집단별 요약은 groupby 다. '무엇으로 나눌지'(Education)를 groupby 에 주고, '무슨 열'(Income·total_spend)의 '무슨 통계'(mean)를 볼지 정한다.
- groupby 에 기준 열을 준 뒤 대괄호로 볼 열 두 개를 고르고 mean 을 붙이면 '집단×통계' 표가 나온다. 위 요구사항에 그 형태가 그대로 적혀 있다.

세부구현:
1. Education 으로 groupby 한다
2. 대괄호로 Income·total_spend 두 열만 고른다
3. mean 으로 평균을 구해 edu_stats 에 담는다
```

</details>

In [ ]:
edu_stats = df.groupby("Education")[["Income", "total_spend"]].mean()
display(edu_stats.round(2))

In [ ]:
# [자가채점]
assert edu_stats.shape == (5, 2)
assert abs(float(edu_stats.loc["PhD", "total_spend"]) - 674.73) < 0.01
assert abs(float(edu_stats.loc["PhD", "Income"]) - 56169.94) < 0.01
assert abs(float(edu_stats.loc["Basic", "total_spend"]) - 81.8) < 0.01
print("✅ 2단계 통과!")

### 해설 — 문제 2 · 2단계
- **접근법**: `groupby('Education')[['Income', 'total_spend']].mean()` 으로 집단별 평균을 한 번에 냅니다.
- **흔한 실수**: 열을 대괄호 **하나**로 넘기면 Series 가, **리스트**로 감싸면 DataFrame 이 나옵니다. 여러 열을 함께 보려면 리스트로 감싸세요.
- **대안**: 평균만으로는 집단 크기를 알 수 없습니다. `size()` 를 함께 보면 표본이 적은 집단(Basic)의 평균이 그만큼 **불안정**하다는 것도 함께 읽을 수 있습니다.

### 3단계 — 결혼상태별 총지출 분포 (박스플롯)
결혼상태(`Marital_Status`)별 `total_spend` 분포를 **박스플롯**으로 비교하세요.

- `sns.boxplot(data=df, x="Marital_Status", y="total_spend")` 로 그리고 제목·축 이름을 답니다.
- 그리기 직전에 `plt.figure()` 를 호출하세요.

이 단계는 **자가채점이 없습니다** — 아래 **완성 그래프(정답)** 처럼 그리면 됩니다.

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day08_기술통계_추론통계/images/과제/lv3_q2_s3.png" width="620"/>

In [ ]:
plt.figure(figsize=(9, 5))
# 박스플롯은 중앙값·사분위·이상치를 한 번에 보여 준다 — 집단끼리 분포를 견줄 때 먼저 그리는 그림이다.
ax = sns.boxplot(data=df, x="Marital_Status", y="total_spend")
ax.set_title("결혼상태별 총지출 분포")
ax.set_xlabel("결혼상태")
ax.set_ylabel("총지출(total_spend)")
plt.show()

### 해설 — 문제 2 · 3단계
- **접근법**: `boxplot` 은 중앙값·사분위·이상치를 한 그림에 담아 **집단 간 분포 비교**에 적합합니다.
- **흔한 실수**: 범주 순서가 **데이터에 등장한 순서**로 정해집니다. 같은 CSV·같은 정제라면 순서도 늘 같지만, 정제 방식이나 데이터가 바뀌면 순서가 달라져 이전 그림과 비교하기 어려워집니다. `order=` 로 고정하면 항상 같은 배열이 됩니다.
- **대안**: 집단별 표본 수 차이가 크면 `violinplot` 이나 `stripplot` 을 겹쳐 실제 분포와 개수를 함께 보여 줄 수 있습니다.

### 4단계 — 캠페인 반응별 총지출 평균 차이 (기술통계)
마지막 캠페인 반응 여부(`Response`, 0=미반응·1=반응)로 나눠 **총지출 평균**을 비교하세요.

- `resp0_mean` = 미반응(0) 그룹의 `total_spend` 평균, `resp1_mean` = 반응(1) 그룹의 평균, `spend_diff` = `resp1_mean - resp0_mean` (반응 그룹이 얼마나 더 쓰는지).

- **요구사항(2자리 반올림)**: `resp0_mean` = **538.85**, `resp1_mean` = **991.24**, `spend_diff` = **452.39**.
- **주의**: 이 단원은 기술통계만 씁니다 — 여기서는 두 집단 평균의 차이만 구합니다(더 깊은 비교는 다음 단원).

<details><summary>힌트</summary>

```text
접근방법:
- 반응 여부로 두 그룹을 나눠 각각 총지출 평균을 구하고 그 차이를 계산한다.

세부구현:
1. Response 가 0 인 행의 total_spend 평균을 구한다
2. Response 가 1 인 행의 total_spend 평균을 구한다
3. 반응 평균에서 미반응 평균을 빼 차이를 구한다
```

</details>

In [ ]:
# 반응한 고객과 아닌 고객의 평균 차이 — 다만 이것만으로는 '많이 써서 반응한 것'인지는 알 수 없다.
resp0_mean = df[df["Response"] == 0]["total_spend"].mean()
resp1_mean = df[df["Response"] == 1]["total_spend"].mean()
# 반응한 고객과 아닌 고객의 평균 차이 — 다만 이것만으로는 '지출이 많아서 반응한 것'인지 알 수 없다.
spend_diff = resp1_mean - resp0_mean
print(round(resp0_mean, 2), round(resp1_mean, 2), round(spend_diff, 2))

In [ ]:
# [자가채점]
assert abs(float(resp0_mean) - 538.85) < 0.01
assert abs(float(resp1_mean) - 991.24) < 0.01
assert abs(float(spend_diff) - 452.39) < 0.01
print("✅ 4단계 통과!")

### 해설 — 문제 2 · 4단계
- **접근법**: 이진 열(`Response`)로 `groupby` 해 평균을 내고 두 값의 차를 구합니다. 이것이 **효과 크기**의 가장 단순한 형태입니다.
- **흔한 실수**: 차이의 **부호를 반대로** 잡기 쉽습니다. '반응 − 미반응' 인지 그 반대인지 지문을 확인하세요.
- **대안**: 이 차이가 우연인지 아닌지 **판정**하는 것이 다음 시간의 가설검정입니다. 오늘은 차이를 '기술'하는 데까지입니다.

### 5단계 — 총지출과 캠페인 수락의 상관
고객이 **여러 캠페인을 수락할수록 더 많이 쓰는지** 를 상관계수로 확인하세요.

1. `accepted_total` = 5개 캠페인 수락 열(['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5']) 의 **행별 합**(0~5) 을 만듭니다.
2. `r_spend_cmp` = `total_spend` 와 `accepted_total` 의 **피어슨 상관계수** — `stats.pearsonr(...)` 의 첫 번째 반환값.

- **요구사항(2자리 반올림)**: `r_spend_cmp` = **0.46** (양의 상관 — 수락 캠페인이 많을수록 지출이 큰 편).
- **주의**: `pearsonr` 은 **두 값**을 돌려줍니다 — **첫 번째(상관계수)** 만 씁니다.

<details><summary>힌트</summary>

```text
접근방법:
- 캠페인 수락 5열을 행 방향으로 더해 수락 개수를 만들고, 총지출과의 피어슨 상관을 구한다.

세부구현:
1. 캠페인 수락 5열을 axis=1 로 합해 accepted_total 을 만든다
2. pearsonr 에 total_spend 와 accepted_total 을 넣는다
3. 반환값의 첫 번째(상관계수)를 r_spend_cmp 에 담는다
```

</details>

In [ ]:
acc_cols = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5']
df["accepted_total"] = df[acc_cols].sum(axis=1)
# 상관은 방향과 세기만 말한다 — 많이 써서 수락한 것인지, 수락해서 많이 쓴 것인지는 이 값으로 못 가른다.
r_spend_cmp, _ = stats.pearsonr(df["total_spend"], df["accepted_total"])
print(round(r_spend_cmp, 2))

In [ ]:
# [자가채점]
assert abs(float(r_spend_cmp) - 0.46) < 0.01
print("✅ 5단계 통과!")

### 해설 — 문제 2 · 5단계
- **접근법**: 여러 캠페인 수락 열을 **행 방향**으로 더해 '수락 횟수'를 만든 뒤, 총지출과의 피어슨 상관을 잽니다.
- **흔한 실수**: `stats.pearsonr` 는 `(r, p)` **두 값**을 돌려줍니다. 첫 번째만 상관계수로 받으세요. 두 열에 결측이 섞여 있으면 에러가 나므로 함께 `dropna` 해야 합니다.
- **대안**: 0.46 은 중간 정도의 양의 상관입니다. **상관은 인과가 아니므로** '캠페인이 지출을 늘렸다'로 읽으면 안 됩니다 — 원래 많이 쓰던 고객이 반응했을 수도 있습니다.

### 6단계 — 인사이트 (서술)
위 집단별 비교와 상관을 근거로 **어떤 고객이 더 많이 쓰고 캠페인에 반응하는지** 를 **3문장 이상** 서술하세요.
- 교육수준·결혼상태별 지출 차이, 반응 그룹의 지출 차이, 수락 캠페인 수와 지출의 관계를 엮어 보세요.

**인사이트 (모범 서술)**

교육수준이 높을수록 지출이 뚜렷하게 커서 PhD 고객의 평균 총지출(675)은 Basic 고객(82)의 여덟 배가 넘고, 평균 소득도 같은 방향으로 높습니다. 박스플롯을 보면 결혼상태별 총지출의 중앙값은 큰 차이가 없지만 어느 집단이든 위쪽으로 긴 꼬리(고지출 고객)가 있어, 지출을 가르는 것은 결혼상태보다 소득·교육 쪽임을 시사합니다. 마지막 캠페인에 **반응한 고객의 평균 총지출(991)은 미반응 고객(539)보다 452 만큼 높고**, 누적 캠페인 수락 수와 총지출의 상관도 0.46 으로 양의 관계여서 — **많이 쓰는 고객이 캠페인에도 더 잘 반응한다**는 마케팅 인사이트를 얻을 수 있습니다.